#UPLOAD SAM3


In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/facebook/sam3

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/facebook/sam3)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login("hf_token")

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("mask-generation", model="facebook/sam3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/538 [00:00<?, ?it/s]

Sam3TrackerModel LOAD REPORT from: facebook/sam3
Key                                                                                          | Status     | 
---------------------------------------------------------------------------------------------+------------+-
tracker_model.memory_attention.layers.{0, 1, 2, 3}.cross_attn_image.o_proj.bias              | UNEXPECTED | 
tracker_model.memory_encoder.mask_downsampler.layers.{0, 1, 2, 3}.layer_norm.bias            | UNEXPECTED | 
tracker_model.memory_attention.layers.{0, 1, 2, 3}.self_attn.o_proj.bias                     | UNEXPECTED | 
tracker_model.mask_decoder.transformer.layers.{0, 1}.self_attn.k_proj.bias                   | UNEXPECTED | 
tracker_model.memory_attention.layers.{0, 1, 2, 3}.self_attn.v_proj.weight                   | UNEXPECTED | 
tracker_model.mask_decoder.transformer.layers.{0, 1}.cross_attn_image_to_token.v_proj.bias   | UNEXPECTED | 
tracker_model.mask_decoder.conv_s1.bias                                        

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("facebook/sam3")
model = AutoModel.from_pretrained("facebook/sam3")

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

#Load Kaggle Pinterest Dataset

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install kagglehub colormath scikit-learn transformers torch torchvision pillow -q
print('✅ Done')

✅ Done


In [ ]:
import kagglehub
from pathlib import Path
import os

# Re-download from Kaggle
dataset_path = Path(kagglehub.dataset_download(
    "anamararullorti/makeup-style-pinterest-web-scrapping-dataset"
))
print(f'Downloaded to: {dataset_path}')

# Check structure
for item in sorted(os.listdir(dataset_path)):
    full = os.path.join(dataset_path, item)
    if os.path.isdir(full):
        print(f'  📁 {item}: {len(os.listdir(full))} files')
    else:
        print(f'  📄 {item}')

# Set DATA_DIR
DATA_DIR = str(dataset_path)
print(f'\nDATA_DIR = "{DATA_DIR}"')

Using Colab cache for faster access to the 'makeup-style-pinterest-web-scrapping-dataset' dataset.
Downloaded to: /kaggle/input/makeup-style-pinterest-web-scrapping-dataset
  📁 clean_makeup: 464 files
  📁 festival_makeup: 489 files
  📁 glam_makeup: 450 files
  📁 no_makeup: 467 files

DATA_DIR = "/kaggle/input/makeup-style-pinterest-web-scrapping-dataset"


# SAM 3 + KAGGLE PINTEREST

In [ ]:
from transformers import Sam3Processor, Sam3Model
from huggingface_hub import login
from google.colab import userdata
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

login("hf_token")

sam_processor = Sam3Processor.from_pretrained('facebook/sam3')
sam_model     = Sam3Model.from_pretrained('facebook/sam3').to(device)
print('✅ SAM3 ready')

Device: cuda


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

✅ SAM3 ready


In [ ]:
from transformers import Sam3Processor, Sam3Model
from huggingface_hub import login
import torch
import os
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.cluster import KMeans
from colormath.color_objects import sRGBColor, LabColor
from colormath.color_conversions import convert_color

# ── Monk Scale ─────────────────────────────────────────────────────────────
MONK_HEX = ['f6ede4','f3e7db','f7ead0','eadaba','d7bd96',
            'a07850','825c43','604134','3a312a','292420']
MONK_RGB  = np.array([[int(h[i:i+2],16) for i in (0,2,4)] for h in MONK_HEX])

def hex_to_monk(hex_str):
    rgb = np.array([int(hex_str[i:i+2],16) for i in (0,2,4)])
    return int(np.argmin(np.sqrt(np.sum((MONK_RGB - rgb)**2, axis=1)))) + 1

def dominant_hex(pixels, k=3):
    if len(pixels) < k:
        k = max(1, len(pixels))
    labs = []
    for p in pixels.astype(np.float32) / 255:
        lab = convert_color(sRGBColor(*p[:3]), LabColor)
        labs.append([lab.lab_l, lab.lab_a, lab.lab_b])
    labs = np.array(labs)
    km = KMeans(n_clusters=k, n_init=5, random_state=42).fit(labs)
    ci = np.argmax(np.bincount(km.labels_))
    L, a, b = km.cluster_centers_[ci]
    rgb = convert_color(LabColor(L, a, b), sRGBColor)
    r  = int(np.clip(rgb.rgb_r * 255, 0, 255))
    g  = int(np.clip(rgb.rgb_g * 255, 0, 255))
    b_ = int(np.clip(rgb.rgb_b * 255, 0, 255))
    return f'{r:02X}{g:02X}{b_:02X}'

# ── Run over full dataset ──────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/makeup-style-pinterest-web-scrapping-dataset"
CATEGORIES = ['no_makeup', 'clean_makeup', 'glam_makeup', 'festival_makeup']

# ── Load existing checkpoint if it exists ─────────────────────────────────
SAVE_PATH = '/content/training_skin_tones.csv'

if os.path.exists(SAVE_PATH):
    existing_df = pd.read_csv(SAVE_PATH)
    records = existing_df.to_dict('records')
    done_files = set(existing_df['filename'].tolist())
    print(f'✅ Loaded checkpoint: {len(records)} records already processed')
else:
    records = []
    done_files = set()
    print('Starting fresh')

total  = 0
failed = 0

for cat in CATEGORIES:
    cat_dir = os.path.join(DATA_DIR, cat)

    if not os.path.isdir(cat_dir):
        print(f'⚠️  {cat} not found, skipping')
        continue

    files = [f for f in os.listdir(cat_dir)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Skip already processed files
    remaining = [f for f in files if f not in done_files]
    print(f'\nProcessing {cat}: {len(remaining)} remaining out of {len(files)} total')

    for fname in remaining:
        total += 1
        fpath = os.path.join(cat_dir, fname)
        try:
            img = Image.open(fpath).convert('RGB')
            img = ImageOps.exif_transpose(img)
            w, h = img.size
            if w > h:
                img = img.rotate(90, expand=True)

            arr = np.array(img)

            inputs = sam_processor(
                images=img,
                text='skin of the face',
                return_tensors='pt'
            ).to(device)

            with torch.no_grad():
                outputs = sam_model(**inputs)

            masks = sam_processor.post_process_instance_segmentation(
                outputs, threshold=0.5, mask_threshold=0.5,
                target_sizes=inputs['original_sizes'].tolist()
            )[0]['masks']

            if len(masks) == 0:
                failed += 1
                continue

            mask   = masks[0].cpu().numpy().astype(bool)
            pixels = arr[mask]
            lum    = pixels.mean(axis=1)
            p10, p90 = np.percentile(lum, 10), np.percentile(lum, 90)
            pixels = pixels[(lum >= p10) & (lum <= p90)]

            if len(pixels) < 3:
                failed += 1
                continue

            hex_val = dominant_hex(pixels)
            monk    = hex_to_monk(hex_val)

            records.append({
                'filename':   fname,
                'category':   cat,
                'skin_hex':   hex_val,
                'monk_level': monk
            })

        except Exception as e:
            print(f'  ⚠️ Failed on {fname}: {e}')
            failed += 1
            continue

    # ── Save checkpoint after each category ───────────────────────────────
    pd.DataFrame(records).to_csv(SAVE_PATH, index=False)
    print(f'  ✅ Done. Successful: {len(records)} total | 💾 Checkpoint saved')

print(f'\n🎉 Finished: {len(records)} successful, {failed} failed out of {total}')

results_df = pd.DataFrame(records)
results_df.to_csv(SAVE_PATH, index=False)
print('\nMonk level distribution:')
print(results_df['monk_level'].value_counts().sort_index())

# ── Download the CSV ───────────────────────────────────────────────────────
from google.colab import files
files.download(SAVE_PATH)
print('✅ CSV downloaded!')

Starting fresh

Processing no_makeup: 467 remaining out of 467 total
  ✅ Done. Successful: 467 total | 💾 Checkpoint saved

Processing clean_makeup: 464 remaining out of 464 total
  ✅ Done. Successful: 927 total | 💾 Checkpoint saved

Processing glam_makeup: 450 remaining out of 450 total
  ✅ Done. Successful: 1377 total | 💾 Checkpoint saved

Processing festival_makeup: 489 remaining out of 489 total
  ✅ Done. Successful: 1866 total | 💾 Checkpoint saved

🎉 Finished: 1866 successful, 4 failed out of 1870

Monk level distribution:
monk_level
1       5
3       2
4      35
5     878
6     659
7     152
8     106
9      24
10      5
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ CSV downloaded!


#Visualization

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# ── Real data ──────────────────────────────────────────────────────────────
monk_counts = {1: 5, 2: 0, 3: 2, 4: 35, 5: 878, 6: 659, 7: 152, 8: 106, 9: 24, 10: 5}

# ── Group into Light / Medium / Dark ──────────────────────────────────────
light  = sum(monk_counts[i] for i in [1, 2, 3])      # MST 1-3
medium = sum(monk_counts[i] for i in [4, 5, 6])      # MST 4-6
dark   = sum(monk_counts[i] for i in [7, 8, 9, 10])  # MST 7-10
total  = light + medium + dark

groups = [light, medium, dark]
labels = [
    f'Light\n(MST 1–3)\nn = {light}',
    f'Medium\n(MST 4–6)\nn = {medium}',
    f'Dark\n(MST 7–10)\nn = {dark}'
]
colors = ['#f7ead0', '#d7bd96', '#604134']
explode = (0.03, 0.03, 0.03)

# ── Figure with donut + legend ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6),
                                gridspec_kw={'width_ratios': [1.3, 1]})
fig.patch.set_facecolor('white')

# ── Left: Donut chart ──────────────────────────────────────────────────────
wedges, texts, autotexts = ax1.pie(
    groups,
    labels=None,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90,
    pctdistance=0.60,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)


autotexts[0].set_position((0.15, 1.1))
autotexts[0].set_color('#5A3A1A')
autotexts[0].set_fontsize(13)
autotexts[0].set_fontweight('bold')

autotexts[1].set_color('#5A3A1A')
autotexts[1].set_fontsize(14)
autotexts[1].set_fontweight('bold')

autotexts[2].set_color('white')
autotexts[2].set_fontsize(14)
autotexts[2].set_fontweight('bold')

# Arrow pointing to the tiny light slice at the top
ax1.annotate('', xy=(0.05, 0.85), xytext=(0.13, 1.05),
             arrowprops=dict(arrowstyle='->', color='#5A3A1A', lw=1.2))


# Centre text
ax1.text(0, 0.08, 'n =', ha='center', va='center',
         fontsize=11, color='#666666')
ax1.text(0, -0.12, '1,866', ha='center', va='center',
         fontsize=16, fontweight='bold', color='#333333')
ax1.text(0, -0.35, 'images', ha='center', va='center',
         fontsize=10, color='#666666')

ax1.set_title('Skin Tone Distribution in the\nPinterest Training Dataset',
              fontsize=13, fontweight='bold', pad=15)

# ── Right: Breakdown per group ─────────────────────────────────────────────
MONK_HEX = ['f6ede4','f3e7db','f7ead0','eadaba','d7bd96',
            'a07850','825c43','604134','3a312a','292420']

mst_levels = list(range(1, 11))
mst_values = [monk_counts[i] for i in mst_levels]
mst_colors = ['#' + h for h in MONK_HEX]

bars = ax2.barh(mst_levels, mst_values,
                color=mst_colors, edgecolor='white',
                linewidth=1, height=0.7)

for bar, val in zip(bars, mst_values):
    if val > 0:
        ax2.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9,
                 fontweight='bold', color='#7A4A4A')

# Group boundary lines
ax2.axhline(3.5, color='#C4836A', linestyle='--', linewidth=1, alpha=0.6)
ax2.axhline(6.5, color='#8B5A5A', linestyle='--', linewidth=1, alpha=0.6)

# Group labels
ax2.text(max(mst_values)*0.6, 2,   'Light',  fontsize=9, color='#C4836A',
         style='italic', fontweight='bold')
ax2.text(max(mst_values)*0.6, 5.5, 'Medium', fontsize=9, color='#8B5A5A',
         style='italic', fontweight='bold')
ax2.text(max(mst_values)*0.6, 8.5, 'Dark',   fontsize=9, color='#4A2A2A',
         style='italic', fontweight='bold')

ax2.set_yticks(mst_levels)
ax2.set_yticklabels([f'MST {i}' for i in mst_levels], fontsize=9)
ax2.set_xlabel('Number of images', fontsize=10)
ax2.set_title('Breakdown by Monk\nSkin Tone Level', fontsize=12,
              fontweight='bold', pad=15)
ax2.set_xlim(0, max(mst_values) * 1.18)
ax2.invert_yaxis()
ax2.spines[['top', 'right']].set_visible(False)
ax2.grid(axis='x', alpha=0.2)

# ── Bottom note ────────────────────────────────────────────────────────────
fig.text(0.5, -0.02,
         'Medium skin tones (MST 4–6) account for 83.4% of the dataset, '
         'reflecting demographic biases in Pinterest user-generated content '
         '(Kärkkäinen and Joo, 2021).',
         ha='center', fontsize=9, style='italic', color='#666666',
         wrap=True)

plt.tight_layout(pad=2.5)
plt.savefig('/content/fig_skin_tone_distribution_donut.png',
            dpi=180, bbox_inches='tight', facecolor='white')

plt.show()

from google.colab import files
files.download('/content/fig_skin_tone_distribution_donut.png')
print('✅ Downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded!
